In [1]:
import pandas as pd
import numpy as np
import json
import os
from os.path import join
from tqdm import tqdm

In [4]:
folder = r'/mnt/c/Users/tre_i/LBNL FDD Datasets/LBNL_FDD_Data_Sets_FCU_all_3/LBNL_FDD_Dataset_FCU'

In [5]:
files = sorted(os.listdir(folder))
files

['FCU_Control_CoolingReverse.csv',
 'FCU_Control_HeatingReverse.csv',
 'FCU_Control_Unstable.csv',
 'FCU_FanOutletBlockage.csv',
 'FCU_FaultFree.csv',
 'FCU_FilterRestriction_10%.csv',
 'FCU_FilterRestriction_20%.csv',
 'FCU_FilterRestriction_50%.csv',
 'FCU_Fouling_Cooling_Airside_Minor.csv',
 'FCU_Fouling_Cooling_Airside_Moderate.csv',
 'FCU_Fouling_Cooling_Airside_Severe.csv',
 'FCU_Fouling_Cooling_Waterside_Minor.csv',
 'FCU_Fouling_Cooling_Waterside_Moderate.csv',
 'FCU_Fouling_Cooling_Waterside_Severe.csv',
 'FCU_Fouling_Heating_Airside_Minor.csv',
 'FCU_Fouling_Heating_Airside_Moderate.csv',
 'FCU_Fouling_Heating_Airside_Severe.csv',
 'FCU_Fouling_Heating_Waterside_Minor.csv',
 'FCU_Fouling_Heating_Waterside_Moderate.csv',
 'FCU_Fouling_Heating_Waterside_Severe.csv',
 'FCU_OABlockage.csv',
 'FCU_OADMPRLeak_20.csv',
 'FCU_OADMPRLeak_50.csv',
 'FCU_OADMPRLeak_80.csv',
 'FCU_OADMPRStuck_0.csv',
 'FCU_OADMPRStuck_100.csv',
 'FCU_OADMPRStuck_30.csv',
 'FCU_OADMPRStuck_50.csv',
 'FCU_

In [6]:
files[0]

'FCU_Control_CoolingReverse.csv'

In [7]:
dfs = {}
for i, file in tqdm(enumerate(files)):
    df = pd.read_csv(join(folder, file), index_col='Datetime')
    dfs[file] = df

0it [00:00, ?it/s]

49it [01:53,  2.32s/it]


In [8]:
import pandas as pd
import hashlib

def df_hash(df):
    # 1. Сортируем колонки
    df = df.sort_index(axis=1)

    # 2. Сортируем индекс
    df = df.sort_index()

    # 3. Округляем числа (чтобы убрать микрошум)
    df = df.round(8)

    # 4. Превращаем в байты
    data_bytes = pd.util.hash_pandas_object(df, index=True).values

    # 5. Считаем общий хэш
    return hashlib.md5(data_bytes).hexdigest()


# создаем словарь для хранения хешей

hashes = {}

for name, data in dfs.items():
    h = df_hash(data)
    hashes.setdefault(h, []).append(name)

# выводим группы одинаковых
for group in hashes.values():
    if len(group) > 1:
        print("Идентичные файлы:")
        for g in group:
            print("  ", g)
        print()

Идентичные файлы:
   FCU_Fouling_Cooling_Airside_Minor.csv
   FCU_Fouling_Heating_Airside_Minor.csv



In [13]:
files_non_repeating = list(set(files) - set(['FCU_Fouling_Heating_Airside_Minor.csv']))
files_non_repeating

['FCU_OADMPRStuck_0.csv',
 'FCU_Fouling_Heating_Airside_Severe.csv',
 'FCU_Fouling_Cooling_Airside_Moderate.csv',
 'FCU_VLVStuck_Heating_20.csv',
 'FCU_VLVLeak_Cooling_50.csv',
 'FCU_VLVStuck_Heating_100.csv',
 'FCU_VLVStuck_Cooling_80.csv',
 'FCU_VLVStuck_Heating_0.csv',
 'FCU_VLVStuck_Heating_80.csv',
 'FCU_Control_CoolingReverse.csv',
 'FCU_FilterRestriction_50%.csv',
 'FCU_Fouling_Cooling_Airside_Minor.csv',
 'FCU_SensorBias_RMTemp_-2C.csv',
 'FCU_FanOutletBlockage.csv',
 'FCU_Control_Unstable.csv',
 'FCU_OABlockage.csv',
 'FCU_Fouling_Cooling_Waterside_Minor.csv',
 'FCU_Fouling_Heating_Waterside_Minor.csv',
 'FCU_VLVStuck_Heating_50.csv',
 'FCU_VLVStuck_Cooling_0.csv',
 'FCU_Fouling_Cooling_Airside_Severe.csv',
 'FCU_VLVLeak_Heating_50.csv',
 'FCU_Fouling_Heating_Waterside_Moderate.csv',
 'FCU_OADMPRStuck_30.csv',
 'FCU_VLVLeak_Heating_20.csv',
 'FCU_SensorBias_RMTemp_+2C.csv',
 'FCU_Fouling_Cooling_Waterside_Moderate.csv',
 'FCU_Fouling_Heating_Waterside_Severe.csv',
 'FCU_Filter

In [12]:
dfs.keys()

dict_keys(['FCU_OADMPRStuck_0.csv', 'FCU_Fouling_Heating_Airside_Severe.csv', 'FCU_Fouling_Cooling_Airside_Moderate.csv', 'FCU_VLVStuck_Heating_20.csv', 'FCU_VLVLeak_Cooling_50.csv', 'FCU_VLVStuck_Heating_100.csv'])

In [16]:
dfs = {}

for i, file in enumerate(tqdm(files_non_repeating)):
    df = pd.read_csv(
        join(folder, file),
        index_col='Datetime',
        parse_dates=['Datetime']  # 🔥 parse once during read
    )
    
    df['fault_type'] = i
    df['target'] = i
    
    dfs[file] = df

# 🔥 do expensive ops ONCE
df_all = pd.concat(dfs, names=["file", "Datetime"])
df_all = df_all.swaplevel(0, 1).sort_index()

  4%|▍         | 2/48 [00:47<18:11, 23.72s/it]


KeyboardInterrupt: 

In [18]:
dfs = {}

for i, file in enumerate(tqdm(files_non_repeating)):
    df = pd.read_csv(join(folder, file), index_col='Datetime')
    df.index = pd.to_datetime(df.index, format="%m/%d/%Y %H:%M")
    df['fault_type'] = i
    df['target'] = i
    df = df.set_index("fault_type", append=True)
    df = df.swaplevel(0, 1).sort_index()
    dfs[file] = df

100%|██████████| 48/48 [02:45<00:00,  3.44s/it]


In [22]:
for k, df in tqdm(dfs.items()):
    dt = df.index.get_level_values(1)   # 1 = Datetime уровень
    assert dt.min() == pd.Timestamp('2018-01-01 00:00:00')
    assert dt.max() == pd.Timestamp('2018-12-31 23:59:00')
    assert df.shape[0] == 365 * 24 * 60

100%|██████████| 48/48 [00:00<00:00, 299.34it/s]


In [29]:
dataset = 'FCU'
fault_types_dict = {i: file[file.find(dataset)+len(dataset)+1:file.find('.csv')] for i, file in enumerate(files)}
fault_types_dict[0] = 'no_fault'

In [30]:
with open('fault_types_dict.json', 'w') as f:
    json.dump(fault_types_dict, f)

In [31]:
df_columns = {k: tuple(v.columns) for k, v in dfs.items()}
len(set(list(df_columns.values())))

1

In [32]:
columns = list(set(list(df_columns.values())))[0]
columns

('FCU_CTRL',
 'FAN_CTRL',
 'RM_TEMP',
 'RMCLGSPT',
 'RMHTGSPT',
 'FCU_MAT',
 'FCU_DAT',
 'FCU_RAT',
 'FCU_CVLV',
 'FCU_CVLV_DM',
 'FCU_CLG_GPM',
 'FCU_CLG_EWT',
 'FCU_CLG_RWT',
 'FCU_HVLV',
 'FCU_HVLV_DM',
 'FCU_HTG_GPM',
 'FCU_HTG_EWT',
 'FCU_HTG_RWT',
 'FCU_DA_CFM',
 'FCU_OA_CFM',
 'FCU_DMPR',
 'FCU_DMPR_DM',
 'FCU_SPD',
 'FCU_OAT',
 'FCU_WAT',
 'FCU_MA_HUMD',
 'FCU_OA_HUMD',
 'FCU_DA_HUMD',
 'FCU_RA_HUMD',
 'target')

In [33]:
df_column_values = {k: {c: set(list(v[c])) for c in columns} for k, v in tqdm(dfs.items())}

100%|██████████| 48/48 [00:57<00:00,  1.21s/it]


In [34]:
for k, v in df_column_values.items():
    print(k)
    print([(col, list(values)[0]) for col, values in v.items() if len(values) == 1 and col != 'target'])

FCU_OADMPRStuck_0.csv
[('FAN_CTRL', 1), ('FCU_CLG_EWT', 45.0), ('FCU_HTG_EWT', 120.0), ('FCU_OA_CFM', 0.88), ('FCU_DMPR', 0.0)]
FCU_Fouling_Heating_Airside_Severe.csv
[('FAN_CTRL', 1), ('FCU_CLG_EWT', 45.0), ('FCU_HTG_EWT', 120.0)]
FCU_Fouling_Cooling_Airside_Moderate.csv
[('FAN_CTRL', 1), ('FCU_CLG_EWT', 45.0), ('FCU_HTG_EWT', 120.0)]
FCU_VLVStuck_Heating_20.csv
[('FCU_CLG_EWT', 45.0), ('FCU_HVLV', 0.2), ('FCU_HTG_GPM', 0.07), ('FCU_HTG_EWT', 120.0)]
FCU_VLVLeak_Cooling_50.csv
[('FAN_CTRL', 1), ('FCU_CLG_GPM', 2.85), ('FCU_CLG_EWT', 45.0), ('FCU_HTG_EWT', 120.0)]
FCU_VLVStuck_Heating_100.csv
[('FAN_CTRL', 1), ('FCU_CLG_EWT', 45.0), ('FCU_HVLV', 1.0), ('FCU_HTG_GPM', 1.04), ('FCU_HTG_EWT', 120.0)]
FCU_VLVStuck_Cooling_80.csv
[('FAN_CTRL', 1), ('FCU_CVLV', 0.8), ('FCU_CLG_GPM', 5.63), ('FCU_CLG_EWT', 45.0), ('FCU_HTG_EWT', 120.0)]
FCU_VLVStuck_Heating_0.csv
[('FCU_CLG_EWT', 45.0), ('FCU_HVLV', 0.0), ('FCU_HTG_GPM', 0.0), ('FCU_HTG_EWT', 120.0)]
FCU_VLVStuck_Heating_80.csv
[('FAN_CTRL', 

In [17]:
features_to_remove = ['OA_CFM', 'SA_SPSPT', 'SA_SP', 'SA_TEMPSPT', 'SF_SPD']

In [18]:
columns_to_keep = list(set(columns) - set(features_to_remove))

In [19]:
dfs_fixed = {k: v[columns_to_keep] for k, v in dfs.items()}

In [20]:
data_full_year = pd.concat([dfs_fixed[file] for file in files_non_repeating if file != 'damper_stuck_100_annual_short.csv'])
data_full_year = data_full_year.sort_index()
data_damper_stuck_100 = dfs_fixed['damper_stuck_100_annual_short.csv']

In [21]:
cut1 = pd.Timestamp("2018-08-01")
cut2 = pd.Timestamp("2018-10-01")

cut_d1 = pd.Timestamp("2018-08-01")
cut_d2 = pd.Timestamp("2018-10-01")

dt_full = data_full_year.index.get_level_values(1)
dt_damper = data_damper_stuck_100.index.get_level_values(1)

train = pd.concat([
    data_full_year[dt_full < cut1],
    data_damper_stuck_100[dt_damper < cut_d1],
]).sort_index()

val = pd.concat([
    data_full_year[(dt_full >= cut1) & (dt_full < cut2)],
    data_damper_stuck_100[(dt_damper >= cut_d1) & (dt_damper < cut_d2)],
]).sort_index()

test = pd.concat([
    data_full_year[dt_full >= cut2],
    data_damper_stuck_100[dt_damper >= cut_d2],
]).sort_index()

In [22]:
train_df = train.drop('target', axis=1)
train_target = train['target']
val_df = val.drop('target', axis=1)
val_target = val['target']
test_df = test.drop('target', axis=1)
test_target = test['target']

In [23]:
write_to = r'/home/ilya_treyvish/projects/lbnl_fdd/data/processed/SDAHU'

In [24]:
train.shape, val.shape, test.shape

((4448700, 26), (1317600, 26), (1899361, 26))

In [25]:
train_df.to_csv(join(write_to, 'train_df.csv'))
print(len(train_df))
train_target.to_csv(join(write_to, 'train_target.csv'))
print(len(train_target))
val_df.to_csv(join(write_to, 'val_df.csv'))
print(len(val_df))
val_target.to_csv(join(write_to, 'val_target.csv'))
print(len(val_target))
test_df.to_csv(join(write_to, 'test_df.csv'))
print(len(test_df))
test_target.to_csv(join(write_to, 'test_target.csv'))
print(len(test_target))

4448700
4448700
1317600
1317600
1899361
1899361


In [26]:
data = pd.concat([data_full_year, data_damper_stuck_100])

In [27]:
data.to_csv(join(write_to, 'full_year_data.csv'))